01. Limpieza Lista de Picking

In [1]:
import os
import pandas as pd
from IPython.display import display
import locale

# Carpeta con los archivos habilitados para macros
carpeta = r"C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Limpieza TiemposRuta Abastos\LP"

# Listar archivos .xlsm en la carpeta
archivos_macros = [f for f in os.listdir(carpeta) if f.endswith('.xlsm')]

# Leer la hoja 'Lista de Picking' del primer archivo encontrado y mostrar solo las columnas A a Q (0 a 16)
if archivos_macros:
    ruta_libro = os.path.join(carpeta, archivos_macros[0])
    try:
        LP = pd.read_excel(ruta_libro, sheet_name="Lista de Picking", header=None, usecols=range(17))
        LP.columns = LP.iloc[2]  # Fila 3 como encabezados (índice 2)
        LP = LP[3:].reset_index(drop=True)  # Eliminar las primeras 3 filas
        # Renombrar columnas
        LP = LP.rename(columns={"Columna6": "PesoDRP(Ton)", "Columna5": "VolumenDRP(m3)"})
        # Agregar columnas Conca CIAT y Conca CIAT COMP
        anio_actual = pd.Timestamp.now().year
        LP['Conca CIAT'] = LP.apply(lambda row: f"{anio_actual}{row['CIAT']}", axis=1) if 'CIAT' in LP.columns else None
        if 'Complemento' in LP.columns:
            LP['Conca CIAT COMP'] = LP.apply(lambda row: f"{anio_actual}{row['Complemento']}" if pd.notna(row['Complemento']) and str(row['Complemento']).strip() != '' else '', axis=1)
        else:
            LP['Conca CIAT COMP'] = None
        # Agregar columnas Año, Mes y Día de la semana en español sin punto final y Día con 3 letras
        if 'Fecha' in LP.columns:
            locale.setlocale(locale.LC_TIME, 'es_ES.UTF-8') if os.name != 'nt' else locale.setlocale(locale.LC_TIME, 'Spanish_Spain')
            fechas = pd.to_datetime(LP['Fecha'], errors='coerce')
            LP['Año'] = fechas.dt.year
            LP['Mes'] = fechas.dt.strftime('%b').str.capitalize().str.replace('.', '', regex=False)
            LP['Día'] = fechas.dt.strftime('%a').str.capitalize().str.replace('.', '', regex=False).str[:3]
    except Exception as e:
        print(f"No se pudo leer la hoja 'Lista de Picking' de {archivos_macros[0]}: {e}")
else:
    print("No se encontraron archivos .xlsm en la carpeta.")

02. Columna Unidad, Mes, Año, Día y Concatenados

In [2]:
import pandas as pd
from IPython.display import display

# Ruta del archivo de Inputs
archivo_inputs = r"C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Monitor Abastos\Concentrado Monitor.xlsx"

# Leer columnas M y N (índices 12 y 13) de la hoja Inputs
tabla7 = pd.read_excel(archivo_inputs, sheet_name="Inputs", usecols=[12, 13], header=None)
tabla7.columns = ["Tipodeunidad", "Unidad"]
tabla7 = tabla7.dropna().reset_index(drop=True)

# Realizar el merge estilo vlookup con LP
if 'LP' in globals():
    LP_merged = LP.merge(tabla7, on="Tipodeunidad", how="left")
    # Convertir columnas Conca CIAT y Conca CIAT COMP a tipo numérico
    if 'Conca CIAT' in LP_merged.columns:
        LP_merged['Conca CIAT'] = pd.to_numeric(LP_merged['Conca CIAT'], errors='coerce')
    if 'Conca CIAT COMP' in LP_merged.columns:
        LP_merged['Conca CIAT COMP'] = pd.to_numeric(LP_merged['Conca CIAT COMP'], errors='coerce')
else:
    print("El DataFrame LP no está disponible. Ejecuta la celda anterior primero.")

03. Envíos

In [3]:
import pandas as pd
from IPython.display import display

# Crear el DataFrame envios a partir de LP_merged
if 'LP_merged' in globals():
    # Filtrar solo las filas donde Tipo es 'Envío'
    envios_group = LP_merged[LP_merged['Tipo'] == 'Envío'].groupby(['CIAT', 'Conca CIAT']).size().reset_index(name='Envios')
else:
    print("El DataFrame LP_merged no está disponible. Ejecuta las celdas anteriores primero.")

04. P-monitor

In [4]:
import pandas as pd
from IPython.display import display

# Crear los DataFrames volumen_rutas_normales, peso_rutas_normales, volumen_rutas_complemento, peso_rutas_complemento y unidad a partir de LP_merged
if 'LP_merged' in globals():
    # Filtrar los datos para rutas normales (solo filtro de Unidad válida)
    filtro_normales = (LP_merged['Unidad'].notna()) & (LP_merged['Unidad'].astype(str).str.strip() != '') & (LP_merged['Unidad'].astype(str).str.strip() != '0')
    volumen_rutas_normales = LP_merged.loc[filtro_normales, ['CIAT', 'Conca CIAT', 'Volumen(m3)', 'Tipo']].copy()
    # Crear el DataFrame peso_rutas_normales con la columna Peso(Kg)
    peso_rutas_normales = LP_merged.loc[filtro_normales, ['CIAT', 'Conca CIAT', 'Peso(Kg)', 'Tipo']].copy()
    # Filtrar los datos para complemento
    filtro_complemento = LP_merged['Complemento'].notna() & LP_merged['Conca CIAT COMP'].notna() & (LP_merged['Conca CIAT COMP'] != '')
    df_complemento_vol = LP_merged.loc[filtro_complemento, ['Complemento', 'Conca CIAT COMP', 'Volumen(m3)']].copy()
    volumen_rutas_complemento = df_complemento_vol.groupby(['Complemento', 'Conca CIAT COMP'], as_index=False)['Volumen(m3)'].sum()
    # Crear el DataFrame peso_rutas_complemento con la columna Peso(Kg)
    df_complemento_peso = LP_merged.loc[filtro_complemento, ['Complemento', 'Conca CIAT COMP', 'Peso(Kg)']].copy()
    peso_rutas_complemento = df_complemento_peso.groupby(['Complemento', 'Conca CIAT COMP'], as_index=False)['Peso(Kg)'].sum()
    # Crear el DataFrame unidad con las columnas CIAT, Conca CIAT y Unidad
    unidad = LP_merged.loc[:, ['CIAT', 'Conca CIAT', 'Unidad']].copy()
    
    # APLICAR DIVISIONES PARA CONVERTIR UNIDADES
    # Volumen: dividir entre 1,000,000 (de mm³ a m³)
    volumen_rutas_normales['Volumen(m3)'] = pd.to_numeric(volumen_rutas_normales['Volumen(m3)'], errors='coerce') / 1000000
    volumen_rutas_complemento['Volumen(m3)'] = pd.to_numeric(volumen_rutas_complemento['Volumen(m3)'], errors='coerce') / 1000000
    
    # Peso: dividir entre 1,000 (de gramos a kg o de kg a toneladas)
    peso_rutas_normales['Peso(Kg)'] = pd.to_numeric(peso_rutas_normales['Peso(Kg)'], errors='coerce') / 1000
    peso_rutas_complemento['Peso(Kg)'] = pd.to_numeric(peso_rutas_complemento['Peso(Kg)'], errors='coerce') / 1000
    
    print("✅ Divisiones aplicadas:")
    print("   - Volumen dividido entre 1,000,000")
    print("   - Peso dividido entre 1,000")
    
else:
    print("El DataFrame LP_merged no está disponible. Ejecuta las celdas anteriores primero.")

✅ Divisiones aplicadas:
   - Volumen dividido entre 1,000,000
   - Peso dividido entre 1,000


05. Reordenar Columnas

In [5]:
# Reordenar las columnas del DataFrame LP_merged según el orden especificado
if 'LP_merged' in globals():
    # Definir el orden deseado de las columnas
    orden_columnas = [
        'Conca CIAT', 'Conca CIAT COMP', 'CIAT', 'Complemento', 'Tipo', 
        'CDR2', 'SART2', 'Fecha', 'Mes', 'Año', 'Día', 'CodigoID', 
        'RutaDRP', 'PesoDRP(Ton)', 'VolumenDRP(m3)', 'Peso(Kg)', 
        'Volumen(m3)', 'Mov.', 'Transporte', 'ValordelBOporpagar', 
        'ValordeMercancia', 'Tipodeunidad', 'Unidad'
    ]
    
    # Verificar qué columnas existen en el DataFrame
    columnas_existentes = LP_merged.columns.tolist()
    
    # Filtrar solo las columnas que existen en el DataFrame
    columnas_ordenadas = [col for col in orden_columnas if col in columnas_existentes]
    
    # Agregar columnas que existen pero no están en el orden especificado
    columnas_adicionales = [col for col in columnas_existentes if col not in orden_columnas]
    if columnas_adicionales:
        print(f"\nColumnas adicionales encontradas (se agregarán al final):")
        for col in columnas_adicionales:
            print(f"  - {col}")
        columnas_ordenadas.extend(columnas_adicionales)
    
    # Reordenar el DataFrame
    LP_merged = LP_merged[columnas_ordenadas]
    
    print(f"DataFrame reordenado exitosamente con {len(LP_merged.columns)} columnas.")
    
else:
    print("El DataFrame LP_merged no está disponible. Ejecuta las celdas anteriores primero.")

DataFrame reordenado exitosamente con 23 columnas.


06. Impresión CSV

In [6]:
from datetime import datetime
from IPython.display import display
import pandas as pd

# Guardar el DataFrame LP_merged en un libro con una sola hoja: Base
if 'LP_merged' in globals():
    fecha_hora = datetime.now().strftime('%Y%m%d_%H%M%S')
    ruta_salida = fr"C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Limpieza TiemposRuta Abastos\Outputs\LP_{fecha_hora}.xlsx"
    with pd.ExcelWriter(ruta_salida) as writer:
        LP_merged.to_excel(writer, sheet_name="Base", index=False)
    print(f"Archivo guardado en: {ruta_salida}")
else:
    print("El DataFrame LP_merged no está disponible. Ejecuta las celdas anteriores primero.")

Archivo guardado en: C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Limpieza TiemposRuta Abastos\Outputs\LP_20260511_101309.xlsx
